# Probe Evaluation Stability v1 — 5組パイロット

既存v1の960文・120 NPZを再利用。CPUのみ、Colab Secrets は GITHUB_RESULTS_TOKEN のみ。Google Drive・OpenAI API・Qwen再推論不要。本実験ではない。評価集合の再サンプリングは新規説明文への一般化を保証しない。正式採択には別の独立集合が必要。

- 64学習文/ラベルで全24層のProbeを固定し、評価8/16/24/32文（4タイプ均衡）を比較。
- 100回の非復元サブサンプリング（32文は全件1回）。同一反復内で入れ子にする。
- AUC、AUC≥0.80判定、平均・母標準偏差(ddof=0)、全評価集合に対する標準化のずれを記録。
- L24のみ、固定Probeの評価ラベルをタイプ内で置換。片側、plus-one、499/999/4999/9999回を入れ子比較。各評価サイズにつき独立乱数10系列（32文の評価集合は固定）。BHはこの5組内の探索的比較のみ。
- 既存結果は変更しない。ペア単位のチェックポイントをGitHubに保存し、再実行時は復元する。

In [ ]:
%pip -q install pandas scikit-learn matplotlib requests tqdm

In [ ]:
import io, json, re, hashlib, base64, time, unicodedata, random
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from google.colab import userdata
from IPython.display import display

REPO="ryouy/GraduationThesis"
SOURCE_REF="328e2c15a0ec64bbbd68112edf95ec7dd31485e3"
SOURCE_BLOB="c0eebc8cbc5b292cab1cc4f7587fb617f16c60b7"
SOURCE="experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/pair_probe_scores.csv"
RUN_ID="20261126_probe_evaluation_stability_v1"
DEST="experiments/2026-11/1126_probe_evaluation_stability/results/"+RUN_ID
MODEL="Qwen/Qwen2.5-0.5B-Instruct"
REV="7ae557604adf67be50417f59c2c2f167def9a775"
API_MODEL="gpt-6-luna"
REASONING_EFFORT="medium"
PAIR_IDS=["C01","C27","C35","C23","C37"]
STYLES=["structure","action","situation","function"]
SIZES=[8,16,32,64]
SEEDS=[42,101,211,307,503]
TRAIN_PER_STYLE=16
EVAL_PER_STYLE=8
GEN_BATCH=8
MAX_ATTEMPTS=8
MAX_API_CALLS=500
FEATURE_CHUNK=8
LAYERS=list(range(1,25))
C=0.1
RUN_API=False
WORK=Path("/content/probe_learning_curve")/RUN_ID
WORK.mkdir(parents=True,exist_ok=True)

def secret(name):
    v=userdata.get(name)
    if not v: raise RuntimeError("Set Colab Secret "+name)
    return v
gh=requests.Session()
gh.headers.update({"Authorization":"Bearer "+secret("GITHUB_RESULTS_TOKEN"),"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"})
GH=f"https://api.github.com/repos/{REPO}/contents/"
def sha(b): return hashlib.sha256(b).hexdigest()
def blob(b): return hashlib.sha1(b"blob "+str(len(b)).encode()+b"\0"+b).hexdigest()
def dump(o): return (json.dumps(o,sort_keys=True,ensure_ascii=False,indent=2,allow_nan=False)+"\n").encode()
def meta(path,ref="main"):
    r=gh.get(GH+path,params={"ref":ref},timeout=90)
    if r.status_code==404:return None
    r.raise_for_status();return r.json()
def get(path,ref="main"):
    m=meta(path,ref)
    if m is None:return None
    if m.get("encoding")=="base64":b=base64.b64decode(m["content"])
    else:
        r=gh.get(GH+path,params={"ref":ref},headers={"Accept":"application/vnd.github.raw+json"},timeout=120)
        r.raise_for_status();b=r.content
    if blob(b)!=m["sha"]:raise RuntimeError("Git blob mismatch "+path)
    return b
def put(path,b):
    # Retry only GitHub writes; never overwrite existing artifacts.
    for attempt in range(8):
        m=meta(path)
        if m is not None:
            if m.get("sha")!=blob(b):raise RuntimeError("Immutable conflict "+path)
            return
        try:
            r=gh.put(GH+path,json={"branch":"main","message":"Probe evaluation stability v1: "+path.rsplit("/",1)[-1],"content":base64.b64encode(b).decode()},timeout=120)
            r.raise_for_status()
            time.sleep(1.2)
            return
        except requests.RequestException as e:
            status=getattr(e.response,"status_code",None)
            if status not in (None,409,429,500,502,503,504) or attempt==7:raise
            time.sleep(min(60,2**(attempt+1)))
def save(name,b):
    put(DEST+"/"+name,b)
    p=WORK/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(b)
def save_df(name,df):save(name,df.to_csv(index=False).encode())
raw=get(SOURCE,SOURCE_REF)
assert raw is not None and blob(raw)==SOURCE_BLOB
all_pairs=pd.read_csv(io.BytesIO(raw))
pairs=all_pairs.set_index("candidate_id").loc[PAIR_IDS].reset_index()
names=sorted(set(pairs.target_x)|set(pairs.comparator_y))
assert len(names)==10 and pairs.category.nunique()==5
display(pairs[["candidate_id","category","target_x","comparator_y","L24_auc"]])
partners={n:set() for n in names}
for r in pairs.itertuples():
    partners[r.target_x].add(r.comparator_y);partners[r.comparator_y].add(r.target_x)
save_df("selected_pairs.csv",pairs)
print("Expected sentences:",len(names)*(TRAIN_PER_STYLE+EVAL_PER_STYLE)*len(STYLES))

## 1. 固定済みデータと特徴量の復元

In [ ]:
PARENT="experiments/2026-11/1126_probe_learning_curve/results/20261126_probe_learning_curve_v1"
PARENT_REF="ddf3d330065399738e2322eaabf03f8790a0d39b"
PARENT_DESC_BLOB="ebd3d8778950954f09f7688ad1f3ec1f5c51eab2"
PARENT_CONFIG_BLOB="b0207cd64df080dc93811a33f202bc3c969072b3"
parent_raw=get(PARENT+"/config.json",PARENT_REF)
assert blob(parent_raw)==PARENT_CONFIG_BLOB
config=json.loads(parent_raw)
assert config["pairs"]==PAIR_IDS and config["layers"]==LAYERS and config["sizes"]==SIZES and config["seeds"]==SEEDS and config["probe_C"]==C
config.update({"parent_ref":PARENT_REF,"parent_run":"20261126_probe_learning_curve_v1","parent_config_blob":PARENT_CONFIG_BLOB,"parent_description_blob":PARENT_DESC_BLOB,"run_id":RUN_ID,"analysis":"evaluation stability","eval_sizes":[8,16,24,32],"repeats":100,"permutation_counts":[499,999,4999,9999],"permutation_repeats":10,"analysis_seed":20261126,"sigma_ddof":0,"auc_gate":0.8,"q_gate":0.05,"permutation":"fixed probe, one-sided, within style","train_per_label":64})
save("config.json",dump(config))
text_raw=get(PARENT+"/descriptions.csv",PARENT_REF)
assert blob(text_raw)==PARENT_DESC_BLOB
texts=pd.read_csv(io.BytesIO(text_raw),keep_default_na=False)
assert len(texts)==960 and texts["text_id"].is_unique
assert set(texts["concept"])==set(names)
assert set(texts["style"])==set(STYLES)
assert set(texts["split"])=={"train","eval"}
counts=texts.groupby(["concept","split","style"]).size()
assert len(counts)==80 and (counts.xs("train",level="split")==16).all() and (counts.xs("eval",level="split")==8).all()
text_hash=sha(text_raw)
parts=[]
for start in tqdm(range(0,len(texts),FEATURE_CHUNK),desc="Restore v1 features"):
    end=min(start+FEATURE_CHUNK,len(texts))
    name=f"features/chunk_{start:04d}.npz"
    b=get(PARENT+"/"+name,PARENT_REF)
    assert b is not None,name
    with np.load(io.BytesIO(b),allow_pickle=False) as z:
        assert int(z["start"])==start and int(z["end"])==end and str(z["text_hash"])==text_hash and z["layers"].tolist()==LAYERS
        a=z["features"].copy()
        assert a.shape==(end-start,24,896) and np.isfinite(a).all()
    parts.append(a)
V=np.concatenate(parts)
assert V.shape==(960,24,896)
print("Verified:",V.shape)
display(counts.unstack("split"))

## 2. 全層の固定Probeと評価集合の安定性

In [ ]:
from scipy.stats import rankdata
EVAL_SIZES=[8,16,24,32]
REPEATS=100
PERM_COUNTS=[499,999,4999,9999]
PERM_REPEATS=10
BASE_SEED=20261126
def checkpoint(name, build):
    path=DEST+"/"+name
    raw=get(path)
    if raw is None:
        df=build()
        raw=df.to_csv(index=False).encode()
        save(name,raw)
    else:
        p=WORK/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)
    return pd.read_csv(io.BytesIO(raw))
def evaluate_pair(p):
    concepts=[p.target_x,p.comparator_y]
    train=np.concatenate([texts.index[(texts["concept"]==n)&(texts["split"]=="train")].to_numpy() for n in concepts])
    test=np.concatenate([texts.index[(texts["concept"]==n)&(texts["split"]=="eval")].to_numpy() for n in concepts])
    yt=np.repeat([1,0],32); ytr=np.repeat([1,0],64)
    styles=texts.loc[test,"style"].to_numpy()
    assert len(train)==128 and len(test)==64
    scores=[]
    for li in range(24):
        sc=StandardScaler().fit(V[train,li])
        clf=LogisticRegression(C=C,solver="liblinear",random_state=42,max_iter=2000).fit(sc.transform(V[train,li]),ytr)
        scores.append(clf.decision_function(sc.transform(V[test,li])))
    scores=np.asarray(scores)
    assert np.isfinite(scores).all()
    out=pd.DataFrame({"candidate_id":p.candidate_id,"text_id":texts.loc[test,"text_id"].to_numpy(),"label":yt,"style":styles})
    for li in range(24):out[f"L{li+1}"]=scores[li]
    return out
def subsets(df, repeat):
    rng=np.random.default_rng(BASE_SEED+repeat)
    orders=[rng.permutation(np.flatnonzero((df.label.to_numpy()==y)&(df["style"].to_numpy()==s))) for y in [1,0] for s in STYLES]
    assert all(len(a)==8 for a in orders)
    return {n:np.concatenate([a[:n//4] for a in orders]) for n in EVAL_SIZES}
def stability(df):
    rows=[]
    labels=df.label.to_numpy()
    for l in LAYERS:
        scores=df[f"L{l}"].to_numpy()
        full_mu=float(scores.mean());full_sd=float(scores.std(ddof=0))
        full_auc=roc_auc_score(labels,scores)
        for rep in range(REPEATS):
            for n,idx in subsets(df,rep).items():
                if n==32 and rep!=0:continue
                ss=scores[idx]; yy=labels[idx]
                mu=float(ss.mean());sd=float(ss.std(ddof=0))
                auc=roc_auc_score(yy,ss)
                valid=np.isfinite(sd) and sd>0 and full_sd>0
                z_rmse=float(np.sqrt(np.mean(((scores-mu)/sd-(scores-full_mu)/full_sd)**2))) if valid else np.nan
                rows.append(dict(candidate_id=df.candidate_id.iloc[0],layer=l,n_eval_per_label=n,repeat=rep,auc=auc,full_auc=full_auc,auc_error=auc-full_auc,auc_pass=auc>=.8,gate_agreement=(auc>=.8)==(full_auc>=.8),mu=mu,sigma=sd,full_mu=full_mu,full_sigma=full_sd,sigma_ratio=sd/full_sd if full_sd>0 else np.nan,relative_mu_error=(mu-full_mu)/full_sd if full_sd>0 else np.nan,z_rmse=z_rmse,valid_sigma=valid))
    return pd.DataFrame(rows)
preds={}; stability_parts=[]
for p in tqdm(list(pairs.itertuples()),desc="Pair checkpoints"):
    pred=checkpoint(f"checkpoints/{p.candidate_id}_predictions.csv",lambda:evaluate_pair(p))
    assert len(pred)==64 and pred.text_id.is_unique and set(pred.candidate_id)=={p.candidate_id}
    preds[p.candidate_id]=pred
    part=checkpoint(f"checkpoints/{p.candidate_id}_stability.csv",lambda:stability(pred))
    assert len(part)==24*(REPEATS*3+1)
    stability_parts.append(part)
st=pd.concat(stability_parts,ignore_index=True)
save_df("evaluation_stability.csv",st)


## 3. 固定Probeの層24置換検定

各タイプ内でX/Yラベル数を維持する。順位和からAUCを計算し、観測値以上の帰無AUCを数える。評価サイズごとの10系列はサブセットと乱数の双方が変化する（32文は乱数のみ）。各系列内の回数比較は同じ置換列の接頭辞を使う。

In [ ]:
def perm_pair(df):
    rows=[]
    for rep in range(PERM_REPEATS):
        for n,idx in subsets(df,rep).items():
            scores=df.L24.to_numpy()[idx];labels=df.label.to_numpy()[idx]
            sty=df["style"].to_numpy()[idx]
            ranks=rankdata(scores,method="average")
            obs=roc_auc_score(labels,scores)
            assert labels.sum()==n
            rng=np.random.default_rng(BASE_SEED+10000+rep*100+n)
            B=max(PERM_COUNTS)
            null_sum=np.zeros(B)
            for s in STYLES:
                r=ranks[sty==s]
                assert len(r)==n//2
                # random uniform subset of n/4 positive ranks
                u=rng.random((B,len(r)))
                chosen=np.argpartition(u,n//4-1,axis=1)[:,:n//4]
                null_sum+=r[chosen].sum(axis=1)
            null_auc=(null_sum-n*(n+1)/2)/(n*n)
            for b in PERM_COUNTS:
                k=int(np.count_nonzero(null_auc[:b]>=obs-1e-12))
                rows.append(dict(candidate_id=df.candidate_id.iloc[0],n_eval_per_label=n,repeat=rep,n_permutations=b,auc=obs,exceedances=k,p=(k+1)/(b+1)))
    return pd.DataFrame(rows)
parts=[]
for cid,pred in tqdm(preds.items(),desc="Permutation checkpoints"):
    part=checkpoint(f"checkpoints/{cid}_permutation.csv",lambda:perm_pair(pred))
    assert len(part)==len(EVAL_SIZES)*PERM_REPEATS*len(PERM_COUNTS)
    parts.append(part)
perms=pd.concat(parts,ignore_index=True)
def bh(p):
    p=np.asarray(p,float); order=np.argsort(p);m=len(p)
    q=np.minimum.accumulate((p[order]*m/np.arange(1,m+1))[::-1])[::-1]
    out=np.empty(m);out[order]=np.minimum(q,1);return out
perms["q"]=perms.groupby(["n_eval_per_label","repeat","n_permutations"])["p"].transform(bh)
perms["pass"]=(perms.auc>=.8)&(perms.q<=.05)
save_df("permutation_stability.csv",perms)


## 4. 集計・図・探索的判断

正式な評価文数やσの閾値は自動決定しない。評価文数32の結果は単一集合であり、ばらつきゼロを意味しない。

In [ ]:
summary=st.groupby(["candidate_id","layer","n_eval_per_label"]).agg(n=("auc","size"),auc_mean=("auc","mean"),auc_sd=("auc","std"),auc_min=("auc","min"),auc_max=("auc","max"),auc_abs_error_mean=("auc_error",lambda x:x.abs().mean()),gate_agreement=("gate_agreement","mean"),mu_error_abs_p95=("relative_mu_error",lambda x:x.abs().quantile(.95)),sigma_ratio_p05=("sigma_ratio",lambda x:x.quantile(.05)),sigma_ratio_p95=("sigma_ratio",lambda x:x.quantile(.95)),z_rmse_p95=("z_rmse",lambda x:x.quantile(.95)),valid_sigma_rate=("valid_sigma","mean")).reset_index()
save_df("stability_summary.csv",summary)
ps=perms.groupby(["candidate_id","n_eval_per_label","n_permutations"]).agg(p_median=("p","median"),p_min=("p","min"),p_max=("p","max"),q_median=("q","median"),pass_rate=("pass","mean")).reset_index()
save_df("permutation_summary.csv",ps)
display(summary[summary.layer==24]);display(ps[ps.n_eval_per_label==32])
def plot_metric(metric,ylabel):
    fig,ax=plt.subplots(figsize=(8,4))
    for cid,g in summary[summary.layer==24].groupby("candidate_id"):
        ax.plot(g.n_eval_per_label,g[metric],marker="o",label=cid)
    ax.set(xlabel="Evaluation texts per label",ylabel=ylabel,xticks=EVAL_SIZES);ax.legend();fig.tight_layout()
    b=io.BytesIO();fig.savefig(b,format="png",dpi=150);save(metric+".png",b.getvalue());plt.show();plt.close(fig)
for metric,label in [("auc_mean","Mean ROC-AUC"),("gate_agreement","Agreement with full-set AUC gate"),("z_rmse_p95","95th percentile Z-score RMSE"),("sigma_ratio_p05","5th percentile SD ratio")]:
    plot_metric(metric,label)
save("analysis_notes.md",("""# Evaluation stability v1
- Exploratory only: five selected pairs; reused evaluation descriptions, not new independent confirmation.
- Fixed 64 training texts per label; all 24 layers; evaluation subsets balanced by style.
- L24 permutation test is conditional on fixed trained probes, one-sided, within style; BH family is five pairs only.
- No formal adoption rule or small-sigma threshold is automatically selected.
- Human factual/semantic audit pending. No main experiment executed.
""").encode())
files={str(p.relative_to(WORK)):sha(p.read_bytes()) for p in sorted(WORK.rglob("*")) if p.is_file() and p.name!="manifest.json"}
save("manifest.json",dump(dict(run_id=RUN_ID,parent_ref=PARENT_REF,n_texts=len(texts),n_stability_rows=len(st),n_permutation_rows=len(perms),human_audit="pending",files=files)))
print("Results:",f"https://github.com/{REPO}/tree/main/{DEST}")
